# Fiorell.IA Final Colab A100 Release

Notebook operativo per chiudere i blocchi residui di Fiorell.IA su runtime Google Colab Pro A100.

Esegue in sequenza:

1. mount Google Drive;
2. uso della root Drive `regulatory-insight-engine` come source of truth operativa;
3. verifica reale CUDA/A100;
4. training finale LoRA `fiorellia_behavior_FINAL_RELEASE`;
5. evaluation reale baseline vs adapter;
6. produzione di `metrics_summary.json`, `comparison.csv`, `final_verdict.md`;
7. test app obbligatori;
8. copia dei risultati finali nella root Drive del repo;
9. commit/push GitHub;
10. avvio Gradio pubblico se il verdetto reale e' `GO DEFINITIVO`.

Non usa Azure e non simula metriche.

In [ ]:
# 00 - Fiorell.IA Drive-first bootstrap
from pathlib import Path
import urllib.request

BOOTSTRAP_REL = "fiorellia_colab_drive_bootstrap.py"
DRIVE_REPO_ROOT = Path("/content/drive/MyDrive/regulatory-insight-engine")
MAC_DRIVE_REPO_ROOT = Path("/Users/itsgennymac/Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/regulatory-insight-engine")
BOOTSTRAP_URL = "https://raw.githubusercontent.com/TheGenesisAIStory/regulatory-insight-engine/main/fiorellia_colab_drive_bootstrap.py"

bootstrap_path = (DRIVE_REPO_ROOT if Path("/content").exists() else MAC_DRIVE_REPO_ROOT) / BOOTSTRAP_REL
bootstrap_path.parent.mkdir(parents=True, exist_ok=True)
if not bootstrap_path.exists() or "drive_first_bootstrap" not in bootstrap_path.read_text(encoding="utf-8", errors="ignore"):
    bootstrap_path.write_text(urllib.request.urlopen(BOOTSTRAP_URL).read().decode("utf-8"), encoding="utf-8")

exec(bootstrap_path.read_text(encoding="utf-8"), globals())


In [ ]:
# 00 - Drive, repo and paths
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = "https://github.com/TheGenesisAIStory/regulatory-insight-engine.git"
COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive")
MAC_DRIVE_REPO_ROOT = Path("/Users/itsgennymac/Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/regulatory-insight-engine")
COLAB_DRIVE_REPO_ROOT = COLAB_DRIVE_ROOT / "regulatory-insight-engine"
TEMP_SOURCE_CLONE = Path("/content/regulatory-insight-engine-source")
GIT_PUBLISH_DIR = Path("/content/regulatory-insight-engine-github")
FINAL_NOTEBOOK_NAME = "fiorellia_final_colab_a100_release.ipynb"

def run(cmd, cwd=None, check=True, env=None):
    print("+", " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, env=env, text=True)

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Drive mount skipped or unavailable: {type(exc).__name__}: {exc}")

DRIVE_REPO_DIR = COLAB_DRIVE_REPO_ROOT if COLAB_DRIVE_ROOT.exists() else MAC_DRIVE_REPO_ROOT
ARTIFACT_DIR = DRIVE_REPO_DIR / "fiorellia-runs" / "final_delivery_latest"

def looks_like_repo(path: Path) -> bool:
    return (path / "fiorellia" / "training" / "final_colab_certification.py").exists()

def git_ok(path: Path) -> bool:
    return subprocess.run(["git", "status", "--short"], cwd=path, text=True, capture_output=True).returncode == 0

def sync_source_to_drive(source: Path, target: Path) -> None:
    target.mkdir(parents=True, exist_ok=True)
    command = [
        "rsync",
        "-a",
        "--exclude=.git/",
        "--exclude=node_modules/",
        "--exclude=.venv-fiorellia-lora/",
        "--exclude=.DS_Store",
        "--exclude=docs/normativa/",
        "--exclude=fiorellia-runs/",
        "--exclude=azure-launch-*/",
        "--exclude=colab-a100-release-*/",
        f"{source}/",
        f"{target}/",
    ]
    run(command)

if not looks_like_repo(DRIVE_REPO_DIR):
    if not (TEMP_SOURCE_CLONE / ".git").exists():
        if TEMP_SOURCE_CLONE.exists():
            shutil.rmtree(TEMP_SOURCE_CLONE)
        run(["git", "clone", REPO_URL, str(TEMP_SOURCE_CLONE)])
    else:
        run(["git", "fetch", "origin", "main"], cwd=TEMP_SOURCE_CLONE)
        run(["git", "reset", "--hard", "origin/main"], cwd=TEMP_SOURCE_CLONE)
    sync_source_to_drive(TEMP_SOURCE_CLONE, DRIVE_REPO_DIR)

repo_candidates = [DRIVE_REPO_DIR, Path.cwd(), TEMP_SOURCE_CLONE]
REPO_ROOT = next((p.resolve() for p in repo_candidates if looks_like_repo(p)), None)

if REPO_ROOT is None:
    raise RuntimeError(f"BLOCCANTE: root Drive non utilizzabile: {DRIVE_REPO_DIR}")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if (REPO_ROOT / ".git").exists() and git_ok(REPO_ROOT):
    run(["git", "status", "-sb"], cwd=REPO_ROOT, check=False)
    pull = run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_ROOT, check=False)
    if pull.returncode != 0:
        print("Git pull did not fast-forward; continuing with the current checkout.")
elif REPO_ROOT == DRIVE_REPO_DIR:
    print("Drive root is source of truth; Git metadata is unavailable here, so GitHub publication will use a clean temporary clone.")

print(json.dumps({
    "repo_root": str(REPO_ROOT),
    "drive_repo_root": str(DRIVE_REPO_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}, indent=2))

In [ ]:
# 01 - Permanent alias/runtime preflight
from pathlib import Path
import shutil

try:
    import yaml  # noqa: F401
except ModuleNotFoundError:
    run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6.0"], cwd=REPO_ROOT)

alias_pairs = [
    (REPO_ROOT / "fiorellia" / "prompts" / "system_prompt.md", REPO_ROOT / "fiorellia" / "prompts" / "system_prompt.txt"),
    (REPO_ROOT / "fiorellia" / "eval" / "eval_set.jsonl", REPO_ROOT / "fiorellia" / "eval" / "eval_set_v0.jsonl"),
    (REPO_ROOT / "fiorellia" / "eval" / "baseline.jsonl", REPO_ROOT / "fiorellia" / "eval" / "prompt_harness_baseline_20260421.jsonl"),
]
for dst, src in alias_pairs:
    if not dst.exists() and src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f"Created runtime alias: {dst.relative_to(REPO_ROOT)}")

for package_init in [
    REPO_ROOT / "fiorellia" / "__init__.py",
    REPO_ROOT / "fiorellia" / "training" / "__init__.py",
    REPO_ROOT / "fiorellia" / "eval" / "__init__.py",
]:
    package_init.parent.mkdir(parents=True, exist_ok=True)
    package_init.touch(exist_ok=True)

required = [
    REPO_ROOT / "fiorellia" / "training" / "final_colab_certification.py",
    REPO_ROOT / "fiorellia" / "training" / "train_lora_behavior_v1.py",
    REPO_ROOT / "fiorellia" / "training" / "configs" / "config_lora_behavior_20260421_style_abstention_patch.yaml",
    REPO_ROOT / "fiorellia" / "training" / "supervised_v1_curated_20260421_style_abstention_patch.jsonl",
    REPO_ROOT / "fiorellia" / "eval" / "prompt_harness_local_adapter.py",
    REPO_ROOT / "fiorellia" / "eval" / "eval_set.jsonl",
    REPO_ROOT / "fiorellia" / "eval" / "baseline.jsonl",
    REPO_ROOT / "fiorellia" / "prompts" / "system_prompt.md",
    REPO_ROOT / "fiorellia_app.py",
    REPO_ROOT / "fiorellia_app_colab.py",
]
missing = [str(p.relative_to(REPO_ROOT)) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing release files: " + ", ".join(missing))
print("Release preflight files: OK")

In [ ]:
# 02 - Mandatory CUDA/A100 verification
import json
import os
import shutil
import subprocess
from pathlib import Path

REQUIRE_A100 = True
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    run([nvidia_smi], cwd=REPO_ROOT)
else:
    print("nvidia-smi non trovato: questo kernel non espone una GPU NVIDIA CUDA.")

import torch

cuda_info = {
    "torch_version": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "device_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
    "devices": [],
    "nvidia_smi": nvidia_smi,
    "colab_tpu_addr": os.getenv("COLAB_TPU_ADDR"),
    "xrt_tpu_config": os.getenv("XRT_TPU_CONFIG"),
    "require_a100": REQUIRE_A100,
}
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        cuda_info["devices"].append({
            "index": index,
            "name": torch.cuda.get_device_name(index),
            "total_memory_gb": round(props.total_memory / (1024 ** 3), 2),
            "major": props.major,
            "minor": props.minor,
        })

print(json.dumps(cuda_info, indent=2))
(ARTIFACT_DIR / "runtime_gpu_preflight.json").write_text(json.dumps(cuda_info, indent=2), encoding="utf-8")

if not cuda_info["cuda_available"]:
    raise RuntimeError(
        "BLOCCANTE: CUDA non disponibile. Il runtime attuale sembra CPU/TPU/TCU: "
        "RAM e disco non bastano per questo training LoRA, serve Runtime type = GPU con A100."
    )
if REQUIRE_A100 and not any("A100" in item["name"] for item in cuda_info["devices"]):
    raise RuntimeError(f"BLOCCANTE: runtime GPU non A100: {cuda_info['devices']}")
if max(item["total_memory_gb"] for item in cuda_info["devices"]) < 35:
    raise RuntimeError("BLOCCANTE: memoria GPU insufficiente per final release.")
print("CUDA/A100 preflight: OK")

In [ ]:
# 03 - Archive stale final outputs before the real run
import shutil
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
archive_dir = ARTIFACT_DIR / f"stale_before_final_{stamp}"
patterns = [
    "metrics_summary*.json",
    "final_verdict*.md",
    "comparison.csv",
    "adapter_eval_scored.jsonl",
    "eval_diagnostics.json",
    "final_certification_summary.json",
    "app_final_test_results.json",
    "app_final_history.jsonl",
    "runtime_gpu.json",
    "final_certification_run.log",
]
moved = []
for pattern in patterns:
    for path in ARTIFACT_DIR.glob(pattern):
        if path.is_file():
            archive_dir.mkdir(parents=True, exist_ok=True)
            target = archive_dir / path.name
            shutil.move(str(path), str(target))
            moved.append(str(target))
print(json.dumps({"archive_dir": str(archive_dir), "moved": moved}, indent=2, ensure_ascii=False))

In [ ]:
# 04 - Final real training + evaluation + app smoke tests
import os
import subprocess
import sys

runner_path = REPO_ROOT / "fiorellia" / "training" / "final_colab_certification.py"
mount_guard = '''def mount_drive_if_colab() -> None:
    drive_path = Path("/content/drive/MyDrive")
    if drive_path.exists():
        print("Drive already available, skipping mount.")
        return

    try:
        from google.colab import drive  # type: ignore
    except Exception:
        print("google.colab non disponibile, skip mount.")
        return

    try:
        import IPython

        if IPython.get_ipython() is None:
            raise RuntimeError("No live IPython kernel available for interactive drive.mount()")
    except Exception as exc:
        raise RuntimeError(
            "Google Drive non montato e mount interattivo impossibile da processo batch. "
            "Monta Drive in una cella notebook prima di lanciare il runner."
        ) from exc

    drive.mount("/content/drive")
'''
runner_text = runner_path.read_text(encoding="utf-8")
if "Drive already available, skipping mount." not in runner_text:
    start = runner_text.index("def mount_drive_if_colab() -> None:")
    end = runner_text.index("\ndef default_artifact_dir()", start)
    runner_text = runner_text[:start] + mount_guard + runner_text[end:]
    runner_path.write_text(runner_text, encoding="utf-8")
runner_text = runner_path.read_text(encoding="utf-8")
if "Drive already available, skipping mount." not in runner_text:
    raise RuntimeError(f"Runner mount guard not applied: {runner_path}")
print(f"Runner mount guard verified: {runner_path}")

cmd = [
    sys.executable,
    "fiorellia/training/final_colab_certification.py",
    "--artifact-dir",
    str(ARTIFACT_DIR),
    "--install-deps",
    "--copy-verdict-to-repo",
]
if not REQUIRE_A100:
    cmd.append("--no-require-a100")
env = os.environ.copy()
env["FIORELLIA_DRIVE_REPO_ROOT"] = str(DRIVE_REPO_DIR)
log_path = ARTIFACT_DIR / "final_certification_run.log"
print("+", " ".join(cmd))
with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
    returncode = proc.wait()

summary_path = ARTIFACT_DIR / "final_certification_summary.json"
if returncode not in (0, 2) or not summary_path.exists():
    raise RuntimeError(f"Final certification failed before producing real summary. returncode={returncode}")
if returncode == 2:
    print("Final certification produced real outputs with NO-GO metrics. Continuing to publish real state.")
else:
    print("Final certification completed with GO DEFINITIVO.")

In [ ]:
# 05 - Verify mandatory final artifacts
import csv
import json

required_outputs = [
    ARTIFACT_DIR / "fiorellia_behavior_FINAL_RELEASE.zip",
    ARTIFACT_DIR / "metrics_summary.json",
    ARTIFACT_DIR / "comparison.csv",
    ARTIFACT_DIR / "final_verdict.md",
    ARTIFACT_DIR / "final_certification_summary.json",
    ARTIFACT_DIR / "app_final_test_results.json",
    ARTIFACT_DIR / "adapter_eval_scored.jsonl",
    ARTIFACT_DIR / "eval_diagnostics.json",
    ARTIFACT_DIR / "reports" / "adapter_eval.jsonl",
]
missing = [str(path) for path in required_outputs if not path.exists()]
if missing:
    raise FileNotFoundError("Missing final artifacts: " + ", ".join(missing))

metrics = json.loads((ARTIFACT_DIR / "metrics_summary.json").read_text(encoding="utf-8"))
for name, value in metrics.items():
    if not isinstance(value, (int, float)) or not 0 <= float(value) <= 1:
        raise RuntimeError(f"Invalid metric {name}={value!r}")

summary = json.loads((ARTIFACT_DIR / "final_certification_summary.json").read_text(encoding="utf-8"))
app_results = json.loads((ARTIFACT_DIR / "app_final_test_results.json").read_text(encoding="utf-8"))
with (ARTIFACT_DIR / "comparison.csv").open("r", encoding="utf-8") as handle:
    comparison_rows = sum(1 for _ in csv.DictReader(handle))

print(json.dumps({
    "verdict": summary.get("verdict"),
    "metrics": metrics,
    "comparison_rows": comparison_rows,
    "app_results_ok": app_results.get("ok"),
    "artifact_dir": str(ARTIFACT_DIR),
}, indent=2, ensure_ascii=False))

In [ ]:
# 06 - Copy real final reports into the repo for GitHub publication
import shutil

repo_report_dir = REPO_ROOT / "fiorellia" / "eval" / "reports" / "final_release"
repo_report_dir.mkdir(parents=True, exist_ok=True)
copy_map = {
    ARTIFACT_DIR / "metrics_summary.json": repo_report_dir / "metrics_summary.json",
    ARTIFACT_DIR / "comparison.csv": repo_report_dir / "comparison.csv",
    ARTIFACT_DIR / "final_certification_summary.json": repo_report_dir / "final_certification_summary.json",
    ARTIFACT_DIR / "app_final_test_results.json": repo_report_dir / "app_final_test_results.json",
    ARTIFACT_DIR / "eval_diagnostics.json": repo_report_dir / "eval_diagnostics.json",
    ARTIFACT_DIR / "adapter_eval_scored.jsonl": repo_report_dir / "adapter_eval_scored.jsonl",
    ARTIFACT_DIR / "final_verdict.md": REPO_ROOT / "fiorellia" / "eval" / "final_verdict.md",
}
copied = []
for src, dst in copy_map.items():
    if not src.exists():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copied.append(str(dst.relative_to(REPO_ROOT)))

print(json.dumps({"copied_to_repo": copied}, indent=2, ensure_ascii=False))

In [ ]:
# 07 - Commit and push the real final state to GitHub
import os
import shutil
import subprocess

PUBLISH_ROOT = REPO_ROOT
if not git_ok(PUBLISH_ROOT):
    if not (GIT_PUBLISH_DIR / ".git").exists():
        if GIT_PUBLISH_DIR.exists():
            shutil.rmtree(GIT_PUBLISH_DIR)
        run(["git", "clone", REPO_URL, str(GIT_PUBLISH_DIR)])
    else:
        run(["git", "fetch", "origin", "main"], cwd=GIT_PUBLISH_DIR)
        run(["git", "reset", "--hard", "origin/main"], cwd=GIT_PUBLISH_DIR)
    for relative in ["fiorellia/eval/final_verdict.md", "fiorellia/eval/reports/final_release"]:
        src = REPO_ROOT / relative
        dst = GIT_PUBLISH_DIR / relative
        if dst.exists():
            shutil.rmtree(dst) if dst.is_dir() else dst.unlink()
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst) if src.is_dir() else shutil.copy2(src, dst)
    PUBLISH_ROOT = GIT_PUBLISH_DIR

run(["git", "config", "user.name", os.getenv("GIT_AUTHOR_NAME", "TheGenesisAIStory")], cwd=PUBLISH_ROOT, check=False)
run(["git", "config", "user.email", os.getenv("GIT_AUTHOR_EMAIL", "actions@users.noreply.github.com")], cwd=PUBLISH_ROOT, check=False)
run(["git", "add", "fiorellia/eval/final_verdict.md", "fiorellia/eval/reports/final_release"], cwd=PUBLISH_ROOT)
cached = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=PUBLISH_ROOT)
if cached.returncode == 0:
    print("No new real final report changes to commit.")
else:
    run(["git", "commit", "-m", "Finalize Fiorell.IA Colab A100 certification"], cwd=PUBLISH_ROOT)

token = os.getenv("GITHUB_TOKEN") or os.getenv("GH_TOKEN")
if token:
    push_url = f"https://{token}@github.com/TheGenesisAIStory/regulatory-insight-engine.git"
    print("+ git push https://***@github.com/TheGenesisAIStory/regulatory-insight-engine.git HEAD:main")
    subprocess.run(["git", "push", push_url, "HEAD:main"], cwd=PUBLISH_ROOT, check=True, text=True)
else:
    run(["git", "push", "origin", "main"], cwd=PUBLISH_ROOT)
run(["git", "status", "-sb"], cwd=PUBLISH_ROOT, check=False)

In [ ]:
# 08 - Launch public Gradio app after GO DEFINITIVO
import json
import subprocess
import sys

summary = json.loads((ARTIFACT_DIR / "final_certification_summary.json").read_text(encoding="utf-8"))
if summary.get("verdict") != "GO DEFINITIVO":
    raise RuntimeError(f"App online bloccata: verdetto reale non positivo: {summary.get('verdict')}")

adapter_dir = Path("/content/fiorellia_behavior_FINAL_RELEASE")
if not adapter_dir.exists():
    raise FileNotFoundError(adapter_dir)

cmd = [
    sys.executable,
    "fiorellia_app_colab.py",
    "--adapter-path",
    str(adapter_dir),
    "--history",
    str(ARTIFACT_DIR / "app_final_live_history.jsonl"),
    "--share",
]
print("+", " ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)